# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print dataset title and description
meta = dataset.metadata
print(meta.name + ": " + meta.description)

## 2. Data Overview
Review available record sets and their fields by `@id`.

In [ ]:
# List all record sets with their @id and names
record_sets = dataset.record_sets()
print("Available Record Sets:")
for rs in record_sets:
    print(f"@id: {rs.id} | Name: {rs.name}")
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for field in rs.fields:
            print(f"    - @id: {field.id} | Name: {field.name} | Data Type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references use the record set and field `@id`s from the above overview.

In [ ]:
# Define the record set(s) to load by @id (replace with discovered IDs from the overview)
# Here we supply the @id from the discovered record sets
record_set_ids = [rs.id for rs in dataset.record_sets()]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from Record Set '@id': {record_set_id}")
        print("Columns:", dataframes[record_set_id].columns.tolist())
        display(dataframes[record_set_id].head())

# For further analysis, pick one record set as main
primary_record_set_id = record_set_ids[0] if record_set_ids else None
if primary_record_set_id is not None:
    print(f"Main analysis will use record set: {primary_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply data processing such as filtering, normalizing, and grouping by attributes. All field references use field `@id`s.

_Note_: Replace the `numeric_field_id` and `group_field_id` below with IDs from the field listing in section 2 for your analysis.

In [ ]:
from numpy import number

# Set these to the relevant field @id values from your chosen record set
record_set_id = primary_record_set_id
if record_set_id is None or record_set_id not in dataframes:
    print("No record set available for analysis.")
else:
    df = dataframes[record_set_id]
    print(f"Fields in {record_set_id}:", list(df.columns))
    # Try to auto-pick a numeric field
    numeric_fields = df.select_dtypes(include='number').columns.tolist()
    if not numeric_fields:
        print("No numeric fields detected.")
    else:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field for EDA: {numeric_field}")

        # Example filter: Only include rows where value > median
        threshold = df[numeric_field].median()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to auto-pick a grouping field
        group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
        group_field = None
        # Pick a different field than numeric
        for gf in group_fields:
            if gf != numeric_field:
                group_field = gf
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and record_set_id in dataframes and numeric_fields:
    df = dataframes[record_set_id]
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}' (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If a group field exists, visualize group means
    if group_field:
        plt.figure(figsize=(10,5))
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values()
        group_means.plot(kind='bar')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(f"{group_field}")
        plt.show()
else:
    print("Insufficient numeric or group fields to plot.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the dataset using its Croissant schema, explored the available record sets and fields by `@id`, extracted records into DataFrames, ran basic filtering and normalization, and visualized select distributions. Further analysis can be performed by selecting different record sets or exploring dataset-specific relationships as appropriate. For custom research use, refer to precise `@id`s of entities as revealed in the record set overview.